# Annotation

Getting a model to label things for you, and checking whether it was right.

1. **The key**
2. **Text**, ten letters to an advice column
3. **Images**, eighteen paintings
4. **Audio**, five recordings from 1894 to 1927

Every dataset downloads itself. Nothing here needs the repo.

> **File → Save a copy in Drive** first.

In [ ]:
%pip install -q -U google-generativeai

In [ ]:
import io, json, os, time
import pandas as pd
import requests
from PIL import Image

def fetch(url, tries=7):
    """Download a URL, waiting and retrying. Big public archives throttle."""
    for n in range(tries):
        try:
            r = requests.get(url, timeout=120, headers={"User-Agent": "culture-as-data course"})
            if r.status_code == 200:
                return r
        except requests.exceptions.RequestException:
            pass
        if n == tries - 1:
            raise RuntimeError(f"gave up on {url}")
        time.sleep(3 * 2 ** n)

os.makedirs("downloads", exist_ok=True)

## 1 · The key

Gemini runs on Google's machines. You reach it with an **API key**, a password tied to your
account.

Three rules: never paste it into a cell, never commit it, keep it in Colab's **Secrets** panel
(the key icon, left) as `GEMINI_API_KEY` with notebook access on. Free keys come from
[aistudio.google.com](https://aistudio.google.com/app/apikey).

With no key everything below still runs on recorded replies.

In [ ]:
API_KEY = os.environ.get("GEMINI_API_KEY")
try:
    from google.colab import userdata
    API_KEY = API_KEY or userdata.get("GEMINI_API_KEY")
except Exception:
    pass
LIVE = bool(API_KEY)
print("live calls:", LIVE, "" if LIVE else "(no key, using recorded replies)")

def ask(parts):
    """Send a prompt (plus any images or audio) and read back one JSON object."""
    import google.generativeai as genai
    genai.configure(api_key=API_KEY)
    raw = genai.GenerativeModel("gemini-2.5-flash").generate_content(parts).text
    return json.loads(raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip())

## 2 · Text

Ten letters to *Dear Abby*, out of the twenty thousand in
[The Pudding's archive](https://github.com/the-pudding/data/tree/master/dearabby). The file is
17 MB and downloads once.

In [ ]:
DA = "https://raw.githubusercontent.com/the-pudding/data/master/dearabby/raw_da_qs.csv"
path = "downloads/dearabby.csv"
if not os.path.exists(path):
    open(path, "wb").write(fetch(DA).content)
abby = pd.read_csv(path)
print(f"{len(abby):,} letters, {int(abby.year.min())} to {int(abby.year.max())}")

# Ten, picked by a phrase each. Nothing distressing: this runs in a classroom.
KEYS = ["keep the ring after the engagement", "gambling problem and someone who is trying",
        "raised with beautiful manners", "dinner table is the only place",
        "bought an airline ticket for him", "comfortable with my body",
        "whether it's a sad or happy occasion", "appropriate christmas gift for an ex-wife",
        "is it ever ok to ask the gender", "been playing the piano for five years"]
text = abby["question_only"].fillna("")
picked = pd.concat([abby[text.str.contains(k, case=False, regex=False)].head(1) for k in KEYS])
letters = [" ".join(str(t).split()) for t in picked["question_only"]]
years = [int(y) for y in picked["year"]]
for y, l in zip(years, letters):
    print(f"{y}  {l[:88]}…")

Label them yourself first. That pass is what you check the model against.

In [ ]:
by_hand = pd.DataFrame({
    "topic": ["romance", "money", "manners", "family", "romance",
              "self", "health", "manners", "manners", "self"],
    "wants": ["validation", "advice", "advice", "advice", "validation",
              "advice", "advice", "advice", "advice", "advice"],
})

TEXT_PROMPT = """Label each letter written to an advice column.
"topic": one of romance, family, money, manners, self, health.
"wants": advice (wants to be told what to do) or validation (wants to be told they were right).
"decade": your best guess at when it was written: 1980s, 1990s, 2000s or 2010s.
Return ONLY a JSON list of objects with keys "topic", "wants", "decade", one per letter, in order.
Letters:
{items}"""

CASSETTE = [
    {"topic": "romance", "wants": "validation", "decade": "1990s"},
    {"topic": "money",   "wants": "advice",     "decade": "2000s"},
    {"topic": "manners", "wants": "validation", "decade": "2000s"},
    {"topic": "family",  "wants": "advice",     "decade": "2010s"},
    {"topic": "money",   "wants": "validation", "decade": "2010s"},
    {"topic": "self",    "wants": "advice",     "decade": "2010s"},
    {"topic": "health",  "wants": "advice",     "decade": "2000s"},
    {"topic": "manners", "wants": "advice",     "decade": "2010s"},
    {"topic": "manners", "wants": "advice",     "decade": "2010s"},
    {"topic": "self",    "wants": "advice",     "decade": "2010s"},
]

prompt = TEXT_PROMPT.format(items="\n".join(f"{i+1}. {x}" for i, x in enumerate(letters)))
said = pd.DataFrame(ask([prompt]) if LIVE else CASSETTE)

print(pd.DataFrame({
    "letter": [l[:34] + "…" for l in letters],
    "topic you": by_hand["topic"], "topic model": said["topic"],
    "wants you": by_hand["wants"], "wants model": said["wants"],
    "written": years, "decade model": said["decade"],
}).to_string(index=False))

In [ ]:
truth = [f"{y // 10 * 10}s" for y in years]
print(f"topic   agrees with you {(by_hand['topic'] == said['topic']).mean():.0%}")
print(f"wants   agrees with you {(by_hand['wants'] == said['wants']).mean():.0%}")
print(f"decade  correct         {(said['decade'] == truth).mean():.0%}\n")
for i in range(len(letters)):
    notes = []
    if by_hand["topic"][i] != said["topic"][i]:
        notes.append(f"topic: you {by_hand['topic'][i]}, model {said['topic'][i]}")
    if by_hand["wants"][i] != said["wants"][i]:
        notes.append(f"wants: you {by_hand['wants'][i]}, model {said['wants'][i]}")
    if said["decade"][i] != truth[i]:
        notes.append(f"decade: really {years[i]}, guessed {said['decade'][i]}")
    if notes:
        print(f"{i+1:2d}. {letters[i][:58]}…")
        for n in notes:
            print("     ", n)

Read the disagreements, not the percentages. Where the model and your pass differ, the label
definition usually did not cover the case. Fix the definition.

## 3 · Images

Same idea, pictures instead. Eighteen portraits from the Met's API: no key, all CC0.

In [ ]:
MET = "https://collectionapi.metmuseum.org/public/collection/v1/objects/"
OBJECTS = [436532, 437397, 436986, 435896, 436896, 436544, 436295, 436623, 435802,
           435581, 435944, 437390, 436658, 437530, 437874, 436840, 437055, 437510]

paintings, titles, made = [], [], []
for oid in OBJECTS:
    meta = fetch(MET + str(oid)).json()
    jpg = f"downloads/{oid}.jpg"
    if not os.path.exists(jpg):
        open(jpg, "wb").write(fetch(meta["primaryImageSmall"]).content)
    paintings.append(Image.open(jpg).convert("RGB"))
    titles.append(meta["title"])
    made.append(meta["objectDate"])
print(len(paintings), "paintings")
paintings[8]                                   # the Bronzino

In [ ]:
# What the Met's own titles say.
sitter = ["man", "man", "woman", "man", "group", "man", "woman", "man", "man",
          "man", "man", "woman", "man", "man", "man", "group", "man", "man"]

IMAGE_PROMPT = """Look at this portrait and answer about the sitter.
"sitter": one of man, woman, group (two or more people).
"century": your best guess at when it was painted, like "16th".
"tell": the single visual detail that decided it, under ten words.
Return ONLY one JSON object with keys "sitter", "century", "tell"."""

CASSETTE_IMG = [
 {"sitter":"man","century":"19th","tell":"straw hat, beard, thick visible brushstrokes"},
 {"sitter":"man","century":"17th","tell":"aged face, dark cap, heavy chiaroscuro"},
 {"sitter":"woman","century":"16th","tell":"white linen headdress and wimple"},
 {"sitter":"man","century":"15th","tell":"white monastic hood, short beard"},
 {"sitter":"group","century":"15th","tell":"two figures framed at a stone window"},
 {"sitter":"man","century":"19th","tell":"dark tailcoat, high white collar"},
 {"sitter":"woman","century":"19th","tell":"dark dress, centre-parted hair"},
 {"sitter":"man","century":"17th","tell":"wide flat white collar, loose brushwork"},
 {"sitter":"man","century":"16th","tell":"black doublet, angular mannerist pose"},
 {"sitter":"man","century":"15th","tell":"red cap, dark cloak, three-quarter view"},
 {"sitter":"man","century":"16th","tell":"gloves held in one hand, flat green ground"},
 {"sitter":"woman","century":"17th","tell":"white ruff and dark Dutch dress"},
 {"sitter":"man","century":"16th","tell":"black cap and gown, letter in hand"},
 {"sitter":"man","century":"17th","tell":"ruff collar, dark doublet, oval format"},
 {"sitter":"man","century":"17th","tell":"long dark hair, plain collar, loose paint"},
 {"sitter":"group","century":"18th","tell":"painter at an easel with two onlookers"},
 {"sitter":"man","century":"15th","tell":"lined face, red hood, plain background"},
 {"sitter":"man","century":"15th","tell":"profile view, red cap, gold ground"},
]

seen = pd.DataFrame([ask([IMAGE_PROMPT, p]) for p in paintings] if LIVE else CASSETTE_IMG)
seen.insert(0, "painting", [t[:32] for t in titles])
seen.insert(1, "truth", sitter)
seen.insert(4, "dated", made)
print(seen.to_string(index=False))
print(f"\nsitter correct on {(seen['truth'] == seen['sitter']).mean():.0%}")

The Bronzino is the one to watch. CLIP, matching the picture against five phrases, called it a
woman. A model that reads the picture has the doublet and the pose to go on.

The "tell" is written after the answer, not before it. Use it to catch a right answer that came
from the wrong evidence.

## 4 · Audio

Five public-domain recordings, 1894 to 1927, from Wikimedia Commons. The year is real ground
truth: the model has to date them by ear.

In [ ]:
RECORDINGS = [
    (1894, "Daisy Bell, Edward M. Favor",
     "https://upload.wikimedia.org/wikipedia/commons/0/07/Daisy_Bell_sung_by_Edward_M._Favor_denoised.ogg"),
    (1904, "African Dreamland, Sousa's Band",
     "https://upload.wikimedia.org/wikipedia/commons/5/5d/African-Dreamland-Sousa_s-Band-_1904_-George-Atwater.ogg"),
    (1912, "Roosevelt, The Liberty of the People",
     "https://upload.wikimedia.org/wikipedia/commons/f/fd/Theodore_Roosevelt_%22The_liberty_of_the_people%22_speech.ogg"),
    (1917, "Livery Stable Blues, Original Dixieland Jass Band",
     "https://upload.wikimedia.org/wikipedia/commons/1/19/Original_Dixieland_Jass_Band_-_Livery_Stable_Blues_%281917%29_with_hiss_reduction.ogg"),
    (1927, "Rhythm Step, Fred Elizalde and his Orchestra",
     "https://upload.wikimedia.org/wikipedia/commons/8/8e/%22Rhythm_Step%22_-_Fred_Elizalde_and_his_Orchestra_%281927%29.opus"),
]

# Commons throttles. Skip what will not come down rather than stopping the notebook.
clips = []
for year, name, url in RECORDINGS:
    path = f"downloads/{year}{os.path.splitext(url)[1]}"
    if not os.path.exists(path):
        try:
            open(path, "wb").write(fetch(url).content)
        except Exception:
            os.path.exists(path) and os.remove(path)
            print(f"{year}  skipped, Commons said no. Re-run this cell later.")
            continue
    clips.append((year, name, path))
    print(f"{year}  {os.path.getsize(path)/1e6:.1f} MB  {name}")

from IPython.display import Audio, display
if clips:
    display(Audio(clips[-1][2]))

In [ ]:
AUDIO_PROMPT = """Listen and answer.
"kind": one of song, band, speech.
"instruments": up to three you can hear, as a list of strings.
"year": your best guess at the year it was RECORDED, as a number.
"tell": the sound that dated it, under ten words.
Return ONLY one JSON object with keys "kind", "instruments", "year", "tell"."""

CASSETTE_AUD = {
 1894: {"kind":"song","instruments":["male voice","piano","brass"],"year":1900,
        "tell":"heavy cylinder hiss, thin narrow-band sound"},
 1904: {"kind":"band","instruments":["cornet","trombone","bass drum"],"year":1905,
        "tell":"acoustic horn recording of a marching band"},
 1912: {"kind":"speech","instruments":["male voice"],"year":1915,
        "tell":"oratorical delivery, flat acoustic capture"},
 1917: {"kind":"band","instruments":["cornet","clarinet","trombone"],"year":1918,
        "tell":"collective improvisation and barnyard effects"},
 1927: {"kind":"band","instruments":["saxophone","piano","drums"],"year":1930,
        "tell":"smooth dance arrangement, electrical recording"},
}

def listen(year, path):
    if not LIVE:
        return CASSETTE_AUD[year]
    blob = {"mime_type": "audio/ogg", "data": open(path, "rb").read()}
    return ask([AUDIO_PROMPT, blob])

heard = pd.DataFrame([listen(y, p) for y, _, p in clips])
heard.insert(0, "recorded", [y for y, _, _ in clips])
heard.insert(1, "what it is", [n[:32] for _, n, _ in clips])
heard["instruments"] = heard["instruments"].apply(", ".join)
print(heard.to_string(index=False))
print(f"\nyear guesses off by {(heard['year'] - heard['recorded']).abs().mean():.0f} years on average")

Kind and instruments it gets. The year it only approximates, and the misses lean late: an 1894
cylinder and a 1917 disc both just sound old.

Everything above ran on replies recorded by hand. Add a key to see what the model really says.

## Three habits

1. Label a sample by hand first, and report the agreement.
2. Read every disagreement. Most are a definition that failed, not a model that did.
3. Count the calls. Ten thousand rows is ten thousand calls unless you batch, and pictures and
   sound cost more per item than text.

### Your turn

Swap in your own corpus. Keep the shape: a fixed label set, a reason with every label, and at
least one label you can mark against something you already know.